# Profiles and simulations

Plot supplied expression profiles and Scanpy’s stored Krumsiek11 simulation. Row order comes from the input or an explicit Scanpy analysis.

Install or update with `pip install --upgrade 'genome-spy-python[scanpy]'`. Choose that environment as your Jupyter kernel.

In [ ]:
import scanpy as sc
import genome_spy.integrations.scanpy as gsc

In [ ]:
pbmc = sc.datasets.pbmc68k_reduced()

## Choose a sample order

Read three markers from the saved raw snapshot and order cells by their stored S-phase score. This is a score ordering, not measured time.

In [ ]:
genes = ["CD79A", "CD3D", "GNLY"]
order = pbmc.obs["S_score"].sort_values(kind="stable").index
profiles = sc.get.obs_df(pbmc, keys=genes, use_raw=True).loc[order]

## Show ordered profiles as a heatmap

Each column is one ordered cell. Zoom both axes to inspect values; numeric labels appear only when they fit.

In [ ]:
profile_heatmap = gsc.pl.timeseries_as_heatmap(
    profiles.to_numpy(),
    var_names=genes,
    color_map="viridis",
    zoom="xy",
    show_values=True,
    width=500,
    height=200,
    show=False,
)
profile_heatmap

## Show values against the score

Keep the actual score on the x-axis. Choose a gene to emphasize it; scroll to zoom and hover for values.

In [ ]:
profile_points = gsc.pl.timeseries(
    profiles,
    time=pbmc.obs.loc[order, "S_score"].to_numpy(),
    var_names=genes,
    xlabel="Stored S-phase score",
    ylabel="Log expression (raw snapshot)",
    marker="o",
    series_select=True,
    ruler=True,
    zoom=True,
    width=500,
    height=250,
    show=False,
)
profile_points

## Compare categorical and numeric colors

Color the same GNLY values by cell-cycle phase or G2M score. Each panel keeps its own colors and zoom.

In [ ]:
options = dict(
    time=pbmc.obs.loc[order, "S_score"].to_numpy(),
    xlabel="Stored S-phase score",
    ylabel="GNLY (log expression)",
    marker="o",
    zoom=True,
    width=250,
    height=220,
    show=False,
)
phase = gsc.pl.timeseries_subplot(
    profiles["GNLY"].to_numpy(),
    color=pbmc.obs.loc[order, "phase"].to_numpy(),
    var_names=sorted(pbmc.obs["phase"].unique()),
    **options,
)
score = gsc.pl.timeseries_subplot(
    profiles["GNLY"].to_numpy(),
    color=pbmc.obs.loc[order, "G2M_score"].to_numpy(),
    var_names=["GNLY"],
    **options,
)
profile_panels = (
    (phase | score)
    .resolve_scale(default="independent")
    .resolve_legend(default="independent")
)
profile_panels

## Load a stored simulation

Krumsiek11 contains four runs of 160 samples across 11 genes. It is simulated data; loading it does not run a new simulation.

In [ ]:
simulation = sc.datasets.krumsiek11()
simulation.obs_names_make_unique()
# H5AD stores label dictionaries with string keys.
simulation.uns["highlights"] = {
    str(k): v for k, v in simulation.uns["highlights"].items()
}

In [ ]:
sim_genes = ["Gata2", "Gata1", "Cebpa", "Pu.1", "Gfi1"]

## Plot simulation runs

Choose a run or gene to emphasize it. Connecting lines stop at run boundaries; scroll to zoom.

In [ ]:
simulation_points = gsc.pl.sim(
    simulation[:, sim_genes],
    tmax_realization=160,
    marker="o",
    run_select=True,
    series_select=True,
    connect=True,
    ruler=True,
    zoom=True,
    width=500,
    height=230,
    show=False,
)
simulation_points

## Show simulation values as a heatmap

Use the same samples and genes. Zoom to read values; the menus emphasize a run or gene without removing samples.

In [ ]:
simulation_heatmap = gsc.pl.sim(
    simulation[:, sim_genes],
    tmax_realization=160,
    as_heatmap=True,
    run_select=True,
    series_select=True,
    ruler=True,
    zoom="xy",
    show_values=True,
    width=500,
    height=230,
    show=False,
)
simulation_heatmap

## Compute a DPT ordering with Scanpy

Use the full simulation to calculate neighbors, diffusion components and pseudotime. Scanpy uses the root sample supplied with the dataset.

In [ ]:
sc.pp.neighbors(simulation, n_neighbors=10, use_rep="X", random_state=0)
sc.tl.diffmap(simulation, random_state=0)
sc.tl.dpt(simulation, n_dcs=10, n_branchings=1)

## Inspect DPT groups and pseudotime

Compare stored branch groups and pseudotime against the saved order. Hover for values; legends identify the groups.

In [ ]:
dpt_groups = gsc.pl.dpt_groups_pseudotime(
    simulation,
    marker="o",
    zoom=True,
    width=500,
    height=170,
    show=False,
)
dpt_groups

## Inspect expression in DPT order

Read the same five genes in the saved DPT order. Zoom to inspect individual values; plotting does not recompute the ordering.

In [ ]:
dpt_expression = gsc.pl.dpt_timeseries(
    simulation[:, sim_genes],
    zoom="xy",
    show_values=True,
    width=500,
    height=230,
    show=False,
)
dpt_expression

:::{admonition} Data use and provenance
:class: note

PBMC profiles use the 700-cell [Scanpy PBMC subset](https://scanpy.scverse.org/en/stable/generated/scanpy.datasets.pbmc68k_reduced.html) from [10x Genomics PBMC68k](https://www.10xgenomics.com/datasets/fresh-68-k-pbm-cs-donor-a-1-standard-1-1-0), licensed [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Credit 10x Genomics and Scanpy’s contributors. S-phase scores supply the displayed order, not experimental time.

[Krumsiek11](https://scanpy.scverse.org/en/stable/generated/scanpy.datasets.krumsiek11.html) is Scanpy’s packaged simulation of a published myeloid-progenitor model. Credit Krumsiek and colleagues and Scanpy’s simulation contributors. Sample names are made unique without changing row order. DPT is computed explicitly above; no simulation is rerun. See the [Scanpy copyright and license](https://github.com/scverse/scanpy/blob/1.11.5/LICENSE). The page reads prepared inputs; the notebook runs the preparation in Python.
:::